# EverFlavor AI - Computer Vision: Food Freshness From Photos

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: pipeline built, waiting for images.** The code is in `src/everflavor/freshness.py` and is tested
> in `tests/test_freshness.py` (label scheme, near-duplicates, the split, coverage, weights and the safety-first
> score). Sources on Mendeley Data download by themselves once their license is read from the dataset record
> (BananaImageBD, AgriFreshNET, FruitNet and Multistage Fish Eyes come by default, about 7 GB); the others
> are unpacked by hand into
> `data/raw/freshness_images/<source>/` after their license is confirmed. Every section runs on what is
> there. Training (section 5) is off by
> default (`TRAIN = False`) because it needs the images and a GPU.

---

## Purpose

The project's **computer vision** pipeline (image classification). From a photo of an ingredient, say
how fresh it looks (fresh, getting old, past its best) and, where the data allows, roughly how long it has
been stored. That is combined with USDA shelf-life guidance from notebook 04.

It covers four food groups, because freshness looks different in each:

| Group | What the camera sees |
|---|---|
| Fruits and vegetables | Color, wrinkles, soft spots, mold |
| Red meat | Color (bright red -> brown / grey), surface dryness or slime |
| Fish | Eyes (clear and bulging -> cloudy and sunken), gills (red -> brown) |
| Bread | Mold spots |

This is a different kind of pipeline from notebooks 01-04: the data is **images**, so it needs its own
collection, labeling, splitting and model training, usually with a GPU (Colab: Runtime > Change runtime
type > GPU).

**Safety rule:** the model only ever says how food **looks**. It never says food is safe to eat: spoilage
and harmful bacteria are often invisible, above all in meat and fish. Every answer ends with "check the
smell, texture and date; when in doubt, throw it out" (the USDA advice), and for meat and fish it adds
the USDA storage time from notebook 04.

## Inputs and outputs

| | File | Made by |
|---|---|---|
| Input | Public image datasets, unpacked by hand into `data/raw/freshness_images/<source>/` (never committed) | Section 2; licenses checked first |
| Input | `data/processed/ingredient_shelf_life.parquet` (USDA storage times) | Notebook 04 (section 5) |
| Output | `data/processed/freshness_images.parquet` (one row per image: label, source, photo set, duplicate group, split, weight) | Notebook 05 (sections 3-4), rebuilt on every run, not stored in GitHub |
| Output | `data/processed/freshness_coverage.csv` (images per group, item, state and source, with warnings) | Notebook 05 (section 4), rebuilt on every run, not stored in GitHub |
| Output | `models/freshness/` (one model per food group and seed) | Notebook 05 (section 5), only when `TRAIN = True`, not stored in GitHub |

## Contents

1. Setup
2. Image Sources and Licenses
3. One Label Scheme for Every Source
4. Cleaning, Coverage and Splitting Without Leakage
5. Model Plan
6. Team Decisions and Next Steps

---

## 1. Setup

The same setup as notebooks 01 and 02: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
# torch and torchvision are already in Colab; locally they are only needed to train (section 5).
%pip install -q requests pandas pyarrow pillow imagehash scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import json
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "calories", "charts", "checks", "cooking", "cuisine", "diets",
                      "environment", "features", "flags", "freshness", "ingredients", "knowledge",
                      "nutrition", "nutrition_quality", "parsing", "pipeline", "progress", "recommend",
                      "reporting", "review", "safety", "sources", "stores", "variants"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

REFRESH_DOWNLOADS = False   # True downloads every source again instead of using the saved copy
SPLIT_FILES = {s: Path(f"data/processed/recipes_{s}.parquet") for s in ["train", "val", "test"]}
ALL_RECIPES_FILE = Path("data/interim/recipes_all.parquet")
IMAGE_DIR = Path("data/raw/freshness_images")   # images are never committed (too large)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_FILE = Path("data/processed/freshness_images.parquet")
COVERAGE_FILE = Path("data/processed/freshness_coverage.csv")
MODEL_DIR = Path("models/freshness")
SHELF_LIFE_FILE = Path("data/processed/ingredient_shelf_life.parquet")

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
missing_inputs = [str(p) for p in [*SPLIT_FILES.values(), ALL_RECIPES_FILE] if not p.exists()]
if missing_inputs:
    print("\nMISSING notebook 01 outputs:", ", ".join(missing_inputs))
    print("Run notebook 01 first (all of Week 5), then run this notebook again.")
else:
    print("Notebook 01 outputs found.")

Runtime        : Local Jupyter (VS Code / Antigravity / JupyterLab)
Python         : 3.12.10
Project folder : ~\Downloads\jupiter-exploration\The-EverFlavor-AI
Notebook 01 outputs found.


In [3]:
from everflavor.freshness import (
    IMAGE_SUFFIXES,
    SAFETY_NOTE,
    STATE_MAP,
    TRAINING_PLAN,
    freshness_coverage,
    group_near_duplicates,
    index_images,
    mendeley_download,
    sample_weights,
    split_without_leakage,
    train_freshness_model,
    zenodo_download,
)

print("Every answer ends with:", SAFETY_NOTE)

Every answer ends with: This only says how the food looks. Check the smell, texture and date; when in doubt, throw it out (USDA).


---

## 2. Image Sources and Licenses

**Team choice: use many sources, even when they overlap.** Photos of the same food taken by different
people, in different countries, years, seasons, kitchens and markets, with different cameras, light and
angles, teach the model what freshness looks like instead of what one lab's table looks like. Overlap is
kept on purpose; only true copies of the same photo are removed (section 4), and each source is weighted
so a large dataset does not drown out a small one (section 5).
Licenses were read on 2026-10-04. **CC BY** means free to use with credit to the authors; **CC BY-SA**
also requires sharing changed versions under the same license. "To confirm" means the license was read
on the article, and the data record itself must be checked before download.

### Fruits and vegetables

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| **AgriFreshNET** (Mendeley 42m5tb7yv9) | 14,500 phone photos, 24 fruits and vegetables, **fresh / semi-fresh / rotten**, with an approximate shelf life per sample (for example 1-4 or 8-12 days); markets, homes, farms | CC BY 4.0 | **Main source:** the middle state, real-world photos, shelf-life ranges |
| **TriModal Ripeness 6** (Figshare 30783827) | Guava, carrot, tomato, Indian gooseberry, banana, mango photographed **3 times a day until they spoil**, plus thermal images and methane readings | CC BY | **Storage time:** exact hours since the start. 19 GB, so only the color photos are downloaded |
| **FruitNet** (Mendeley b6fftwbr2v) | 14,700 photos of 6 Indian fruits, good / bad quality, different backgrounds and light | CC BY 4.0 | More examples of 6 common fruits |
| **BananaImageBD** (Mendeley ptfscwtnyz) | Bananas in 4 stages: green, semi-ripe, ripe, overripe (820 original photos) | CC BY 4.0 | Ripeness stages for the most common fruit |
| Fruits-360 (Kaggle) | 250+ classes, studio photos on white | CC BY-SA 4.0 | Item recognition only (which fruit is it), not freshness |

### Red meat

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| **MeatScan** (Zenodo 10.5281/zenodo.16764338) | 6,995 original photos of beef, fresh vs spoiled, in markets, butcher shops and cold storage; 3 labelers agreed (Fleiss' kappa 0.87) | CC BY (to confirm) | Real market photos with checked labels |
| Meat freshness (Roboflow Universe) | About 2,000 photos, fresh / half-fresh / spoiled | CC BY 4.0 (to confirm) | The middle state for meat |

### Fish

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| **DaFiF** (Mendeley vx4ptwk3pb) | Mackerel, tilapia, tuna photographed daily for **11 days**, with an expert freshness score (1-9) for eyes, gills, skin, flesh, smell and texture; also gas sensor readings | CC BY 4.0 | **Storage time and expert scores**, not just fresh / spoiled |
| **Freshness of the Fish Eyes** (Mendeley xzyx7pbr3w) | 4,392 eye photos, 8 species, days 1-2 / 3-4 / 5-6 | CC BY 4.0 | Many species; day ranges |
| **Multistage Fish Eyes** (Mendeley 67nmx3mhwh) | 4,800 eye photos, 2 freshwater species, 4 light and background setups, days ≤2 / 2-4 / >4 | CC BY 4.0 | Balanced classes, varied lighting |

### Bread

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| Bread mold detection (Roboflow Universe) | About 900 photos with mold marked by boxes | CC BY 4.0 (to confirm) | Mold, the main sign for baked goods |

### Our own photos

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| Team photos | Groceries bought locally, photographed every day in a real kitchen | Team consent | **The final test:** most public photos were taken in India, Bangladesh, Pakistan, Ghana, Indonesia and China, so this checks the model on local produce, cameras and kitchens |

**Not used:** datasets with no stated license, and augmented copies inside a dataset (MeatScan and
BananaImageBD ship rotated and flipped copies). Only original photos are kept, so copies of one photo
cannot end up in both training and test.

In [4]:
# Sources on Mendeley Data download here, after their license is read from the dataset record
# (mendeley_download refuses anything but CC BY / CC0). About 7 GB for the five below (once; reused after);
# DaFiF (8 GB) is left out until its separate label file is read. MeatScan (Zenodo, 25 GB, beef) is opt-in:
# set DOWNLOAD_MEATSCAN = True to download it (needs about 50 GB free while it unpacks).
MENDELEY = {"agrifreshnet": ("42m5tb7yv9", ""), "fruitnet": ("b6fftwbr2v", ""),
            "bananaimagebd": ("ptfscwtnyz", "Ripeness Detection"), "dafif": ("vx4ptwk3pb", ""),
            "fish_eyes_ffe": ("xzyx7pbr3w", ""), "fish_eyes_mfed": ("67nmx3mhwh", "")}
MENDELEY["fruitnet"] = ("b6fftwbr2v", "FruitNetDataset")   # the same photos also ship as a .rar
DOWNLOAD_SOURCES = ["bananaimagebd", "agrifreshnet", "fruitnet", "fish_eyes_mfed", "fish_eyes_ffe"]
for source in DOWNLOAD_SOURCES:
    dataset_id, name_contains = MENDELEY[source]
    mendeley_download(dataset_id, IMAGE_DIR / source, name_contains, refresh=REFRESH_DOWNLOADS)
DOWNLOAD_MEATSCAN = False
if DOWNLOAD_MEATSCAN or (IMAGE_DIR / "meatscan" / "LICENSE.txt").exists():
    zenodo_download("16764338", IMAGE_DIR / "meatscan", refresh=REFRESH_DOWNLOADS)   # reads its license first

IMAGE_SOURCES = pd.DataFrame([
    # source, food group, host, license, labels, storage time known
    ("agrifreshnet", "produce", "mendeley 42m5tb7yv9", "CC BY 4.0", "fresh/semi-fresh/rotten", "shelf-life range"),
    ("trimodal_ripeness6", "produce", "figshare 30783827", "CC BY", "time series", "hours"),
    ("fruitnet", "produce", "mendeley b6fftwbr2v", "CC BY 4.0", "good/bad", "no"),
    ("bananaimagebd", "produce", "mendeley ptfscwtnyz", "CC BY 4.0", "green/semi-ripe/ripe/overripe", "no"),
    ("meatscan", "meat", "zenodo 16764338", "CC BY 4.0", "fresh/spoiled (beef)", "no"),
    ("roboflow_meat", "meat", "roboflow universe", "CC BY 4.0 (to confirm)", "fresh/half-fresh/spoiled", "no"),
    ("dafif", "fish", "mendeley vx4ptwk3pb", "CC BY 4.0", "expert score 1-9", "days"),
    ("fish_eyes_ffe", "fish", "mendeley xzyx7pbr3w", "CC BY 4.0", "highly fresh/fresh/not fresh", "no"),
    ("fish_eyes_mfed", "fish", "mendeley 67nmx3mhwh", "CC BY 4.0", "days <=2/2-4/>4", "day range"),
    ("roboflow_bread_mold", "bread", "roboflow universe", "CC BY 4.0 (to confirm)", "mold boxes", "no"),
    ("team_photos", "all", "team", "team consent", "day by day", "days"),
], columns=["source", "group", "host", "license", "labels", "storage_time"])
IMAGE_SOURCES["folder"] = IMAGE_SOURCES["source"].map(lambda s: str(IMAGE_DIR / s))
IMAGE_SOURCES["images_found"] = IMAGE_SOURCES["folder"].map(
    lambda f: sum(1 for p in Path(f).rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES) if Path(f).is_dir() else 0)
print(IMAGE_SOURCES.drop(columns=["folder"]).to_string(index=False))
print(f"\nOther sources: unpack into {IMAGE_DIR}/<source>/ (the 'source' column) after confirming the license.")

             source   group                host                license                        labels     storage_time  images_found
       agrifreshnet produce mendeley 42m5tb7yv9              CC BY 4.0       fresh/semi-fresh/rotten shelf-life range         14160
 trimodal_ripeness6 produce   figshare 30783827                  CC BY                   time series            hours             0
           fruitnet produce mendeley b6fftwbr2v              CC BY 4.0                      good/bad               no         19526
      bananaimagebd produce mendeley ptfscwtnyz              CC BY 4.0 green/semi-ripe/ripe/overripe               no           820
           meatscan    meat     zenodo 16764338              CC BY 4.0          fresh/spoiled (beef)               no         11000
      roboflow_meat    meat   roboflow universe CC BY 4.0 (to confirm)      fresh/half-fresh/spoiled               no             0
              dafif    fish mendeley vx4ptwk3pb              CC BY 4.0      

---

## 3. One Label Scheme for Every Source

Every source labels differently, so each is mapped to one scheme (like cuisine families in notebook 01).
`index_images` reads the state, the item and any storage age from each image's folder and file names
("Rotten/Apple/12.jpg", "tilapia_day3/4.jpg", "carp_1-2days/7.jpg"); a source's own augmented copies
("aug", "flip", "rotate" in the name) are skipped, and images whose state can not be read are kept with an
empty state for a hand check.
Two targets: the **state** (every image has one) and the **storage age** (only images whose source
records it).

| Column | Values |
|---|---|
| `group` | `produce`, `meat`, `fish`, `bread` |
| `item` | Our ingredient name (`apple`, `banana`, `beef`, `tilapia` ...) |
| `state` | `fresh`, `aging`, `spoiled` |
| `age_days_min`, `age_days_max` | Storage time when the source records it (an exact day gives min = max); empty otherwise |
| `expert_score` | DaFiF's 1-9 freshness score; empty for other sources |
| `photo_set` | One fish, one fruit or one photo session: all its photos stay in one split |
| `source`, `license` | Where the image came from |

The state is mapped from each source's own words. Sources that only give storage days (the fish eye
sets) get a state from the day ranges, and that rule is written down so the team can change it.

In [5]:
print("State words ->", ", ".join(f"{k}: {v}" for k, v in STATE_MAP.items()))
# Team decision: is an overripe banana "aging" or "spoiled"? (section 6)

found = IMAGE_SOURCES[IMAGE_SOURCES["images_found"] > 0]
groups = {"fish_eyes_ffe": "fish", "fish_eyes_mfed": "fish", "dafif": "fish", "meatscan": "meat",
          "roboflow_meat": "meat", "roboflow_bread_mold": "bread"}
# Sources that photograph one item over time: each folder is one photo set (kept in one split)
series_sources = {"trimodal_ripeness6", "dafif", "team_photos"}
# Team rule (section 6): sources whose words mean something else. Multistage Fish Eyes labels by days in storage:
# "Highly Fresh" (2 days or less) -> fresh, "Fresh" (2-4 days) -> aging, "Not Fresh" (over 4 days) -> spoiled.
# Fish Eyes (FFE) uses the same three words, read the same way.
FISH_EYES_RULE = {"highly fresh": "fresh", "fresh": "aging", "not fresh": "spoiled"}
SOURCE_STATE_RULES = {"fish_eyes_mfed": FISH_EYES_RULE, "fish_eyes_ffe": FISH_EYES_RULE}
images = pd.concat([index_images(r.source, r.folder, r.license, groups.get(r.source), r.source in series_sources,
                                 SOURCE_STATE_RULES.get(r.source)) for r in found.itertuples()],
                   ignore_index=True) if len(found) else pd.DataFrame()
if images.empty:
    print("\nNo images yet: download a source (section 2) into its folder, then run this notebook again.")
else:
    print(f"\n{len(images):,} images from {images['source'].nunique()} sources; "
          f"state read for {images['state'].notna().mean():.0%}")
    print(images.groupby(["group", "state"], dropna=False).size().unstack(fill_value=0))
    print("Images per source:", images["source"].value_counts().to_dict())

State words -> fresh: fresh, good: fresh, highly fresh: fresh, green: fresh, semi-ripe: fresh, semiripe: fresh, semi ripe: fresh, unripe: fresh, semi-fresh: aging, semifresh: aging, semi fresh: aging, half-fresh: aging, halffresh: aging, half fresh: aging, ripe: aging, overripe: aging, medium: aging, rotten: spoiled, bad: spoiled, spoiled: spoiled, not fresh: spoiled, stale: spoiled, moldy: spoiled, mouldy: spoiled


meatscan: skipped 439 files that are not images (empty or broken)



45,314 images from 6 sources; state read for 98%
state    aging  fresh  spoiled   NaN
group                               
fish      2920   3364     2906     0
meat         0   5627     4934     0
produce   2064  14014     8411  1074
Images per source: {'fruitnet': 19526, 'meatscan': 10561, 'agrifreshnet': 5217, 'fish_eyes_mfed': 4800, 'fish_eyes_ffe': 4390, 'bananaimagebd': 820}


---

## 4. Cleaning, Coverage and Splitting Without Leakage

- **Duplicates:** near-identical photos (the same fruit from two angles, or a dataset that copies
  another) are found with a perceptual hash and kept in one split only, like `ingredient_group` in
  notebook 01.
- **Split by `photo_set`**, never by single image: DaFiF photographs the same fish for 11 days, so a
  random split would put day 3 of a fish in training and day 4 of the same fish in test.
- **Hold out whole sources** as an extra test: train without one source and test on it, to see whether
  the model works on photos from a place and camera it has never seen.
- **Coverage report** (`freshness_coverage.csv`): images per group, item, state and source. It shows which
  items have too few photos, the same way notebook 03 reports coverage by country.

In [6]:
if images.empty:
    print("No images yet: duplicates, split and coverage run once a source is downloaded.")
else:
    images["duplicate_group"] = group_near_duplicates(images, max_distance=4)
    images["split"] = split_without_leakage(images)
    labeled = images["state"].notna()
    images.loc[labeled, "weight"] = sample_weights(images[labeled])
    images.to_parquet(IMAGES_FILE, index=False)
    coverage = freshness_coverage(images, min_images=100)
    coverage.to_csv(COVERAGE_FILE, index=False)
    print(f"Near-duplicate groups: {images['duplicate_group'].nunique():,} for {len(images):,} images")
    print(images["split"].value_counts().to_string())
    print(f"Saved {IMAGES_FILE} and {COVERAGE_FILE}; items and states with fewer than 100 images:")
    print(coverage[coverage["warning"] != ""][["group", "item", "state", "item_state_images"]]
          .drop_duplicates().to_string(index=False))

Perceptual hashes:   0%|          | 0/45314 [00:00<?, ?it/s]

Near-duplicate groups: 30,491 for 45,314 images
split
train    31898
test      6780
val       6636
Saved data\processed\freshness_images.parquet and data\processed\freshness_coverage.csv; items and states with fewer than 100 images:
  group      item   state  item_state_images
   fish threadfin   aging                 80
   fish threadfin   fresh                 80
   fish threadfin spoiled                 80
produce  cucumber spoiled                 47
produce    papaya   aging                 49


---

## 5. Model and Training Plan

1. **One model per food group** to start (produce, meat, fish, bread): a fish eye and a strawberry share
   little, and a small model per group is easier to check.
2. Fine-tune a small pretrained image model (for example MobileNet or EfficientNet from `torchvision` or
   `timm`) on `state`.
3. Where storage age is known, a second output predicts the age range ("about 3-4 days"). It is trained
   only on images that have an age.
4. **Report:** accuracy per item, a confusion matrix, and above all how often **spoiled food is called
   fresh** (the costly error). The decision threshold is set to keep that error low, even if more fresh
   food gets a "check it" warning.
5. **Test** on held-out sources and on the team's own photos before any agent uses the model.

### How many epochs?

No fixed number: the **validation split decides when to stop**.

| Phase | What trains | Epochs | Learning rate |
|---|---|---|---|
| 1. Head only | Only the new last layer; the pretrained model is frozen | About 5 | Normal (for example 1e-3) |
| 2. Fine-tuning | The whole model | Up to about 30, with **early stopping** | Lower (for example 1e-4), reduced when validation stops improving |

- **Early stopping:** after every epoch the model is scored on the validation split. If the score has not
  improved for 5 epochs (`PATIENCE`), training stops and the **best** epoch is kept, not the last one.
  Fine-tuning on datasets of this size usually settles within 10-20 epochs.
- **The score that picks the best epoch** puts safety first: the share of spoiled photos called fresh,
  then macro F1 (accuracy that counts every state equally).
- **Balance by source:** each photo is weighted so every source counts about equally in training, and
  within a source every state counts equally.
- **Augmentation during training** (random crops, flips, small color and brightness changes) helps the
  model cope with other cameras and light. It is applied on the fly, never saved as extra files.
- **The test split is used once, at the very end.** Checking it after every change would tune the model
  to it, and the final number would look better than the model really is.
- **Repeat with 3 random seeds** and report the average and spread, so one lucky run is not mistaken for
  a good model.

In [7]:
TRAIN = False   # True trains one model per food group and seed (needs the images and, in practice, a GPU)
print(json.dumps(TRAINING_PLAN, indent=2))

if TRAIN and not images.empty:
    results = [train_freshness_model(images[images["state"].notna()], group, MODEL_DIR, TRAINING_PLAN, seed)
               for group in sorted(images["group"].dropna().unique()) for seed in TRAINING_PLAN["seeds"]]
    scores = pd.DataFrame([{"group": r["group"], "seed": r["seed"], "best_epoch": r["best_epoch"], **r["val_scores"]}
                           for r in results])
    print(scores.groupby("group")[["spoiled_called_fresh", "macro_f1", "accuracy"]].agg(["mean", "std"]).round(3))
else:
    print("\nTraining is off (TRAIN = False) or there are no images yet.")

{
  "image_size": 224,
  "batch_size": 32,
  "head_only_epochs": 5,
  "max_finetune_epochs": 30,
  "patience": 5,
  "head_learning_rate": 0.001,
  "finetune_learning_rate": 0.0001,
  "selection_metric": "spoiled_called_fresh (lower is better), then macro F1",
  "weights": "equal per source, then equal per state within a source",
  "seeds": [
    0,
    1,
    2
  ]
}

Training is off (TRAIN = False) or there are no images yet.


---

## 6. Team Decisions and Next Steps

1. **Pilot items:** a short list per group, picked from the coverage report.
2. **State rules:** is an overripe banana "aging" or "spoiled"? How many storage days make fish "aging"?
3. **Team photos:** who takes them, how often, and a fixed routine (same light, same angle, date in the
   file name).
4. **Licenses marked "to confirm":** check the data record before download.

**Build order:** produce first (AgriFreshNET, FruitNet), then fish (the three Mendeley sets), then meat,
then bread.

---

## Plan status

What is built and what is waiting for images, licenses or a GPU.

In [8]:
PLAN = pd.DataFrame([
    ("2 Image sources (11, 4 food groups)", f"{int((IMAGE_SOURCES['images_found'] > 0).sum())} of {len(IMAGE_SOURCES)} downloaded; 3 licenses to confirm"),
    ("3 Image index and label scheme", "built" + (f": {len(images):,} images" if not images.empty else "; waiting for images")),
    ("4 Duplicates, coverage and split", "built" + ("" if not images.empty else "; waiting for images")),
    ("5 Models (one per food group)", "built; TRAIN = False until images and a GPU are ready"),
], columns=["step", "status"])
print(PLAN.to_string(index=False))

                               step                                                status
2 Image sources (11, 4 food groups)             6 of 11 downloaded; 3 licenses to confirm
     3 Image index and label scheme                                  built: 45,314 images
   4 Duplicates, coverage and split                                                 built
      5 Models (one per food group) built; TRAIN = False until images and a GPU are ready
